# Simple GNN training example

This notebook trains the repository's edge-feature GNN on a prepared bootstrap graph. Before running it, use `gufu.py` cells 1–4 in `RUN_MODE = "stepwise"` to prepare the scans, train or load the AE, and build a matching `GraphState`. Then point `graph_path` below at that graph. Fit the initial reference-site transform first with `python spatial_normalization.py fit --input input/GF/fingerprint0.txt --site-id GF --output artifacts/spatial/GF.json --coordinate-units unknown`; `gufu.py` loads and records that frozen transform.

Virtual positive pairs and negative scan pairs are mined from normalized RSSI cosine similarity, using the thresholds recorded in the graph. The GNN loss then attracts the endpoints of physical and virtual edges and repels the mined negative pairs in learned embedding space. The optimizer loop uses the same `equation4_loss` function as `sage_ve.train_gnn`. Nothing is saved to disk.

The positive edge term follows the paper's Eq. (4) form with the implementation's finite log clamp and leaky-ReLU choice. The added negative term uses a margin on learned embedding cosine, weighted by RSSI-derived negative strength. RSSI similarity selects the training pairs; it is not itself the learned-embedding loss.


In [1]:
from pathlib import Path
import torch
import torch.nn.functional as F

from graph_build import load_graph_state
from sage_ve import (
    EdgeFeatureSAGE,
    GNNModelConfig,
    GNNTrainingConfig,
    equation4_loss,
    graph_state_to_tensors,
    equation4_loss_components
)

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# This is the current gufu.py real-data run. Change it to the graph_state.json
# produced by cells 1–4 if you used a different AE/configuration/run directory.
graph_path = Path(
    "output/notebook_runs/GF/real-GF-fingerprint0-ae-v2-mse-rssi-pairs-v1-spatial-v1-softplus-cosine-t0p1-nw1-d256-z32-drop0p1/graph_state.json"
)
if not graph_path.is_file():
    raise FileNotFoundError(
        f"Missing {graph_path}. Run gufu.py cells 1–4 in stepwise mode, then set graph_path to that run's graph_state.json."
    )

graph = load_graph_state(graph_path)
if graph.batch_role != "bootstrap_reference":
    raise ValueError("This example trains only on the labeled bootstrap graph.")
graph_tensors = graph_state_to_tensors(graph, device=device, dtype=torch.float32)
node_count = graph_tensors.node_features.shape[0]
edge_count = graph_tensors.edge_features.shape[0]
if edge_count == 0:
    raise ValueError("Equation (4) needs at least one physical or virtual edge.")
assert graph_tensors.node_features.shape == (node_count, graph.node_feature_dim)
assert graph_tensors.edge_features.shape == (edge_count, graph.node_feature_dim)
assert len(graph_tensors.node_ids) == node_count
assert len(graph_tensors.edge_ids) == edge_count

print(f"graph: {graph_path}")
print(f"dataset/batch: {graph.dataset_id}/{graph.batch_id}")
print(f"graph ID: {graph.graph_id}")
print(f"normalization: {graph.normalization}")
if not graph.build_config.spatial_normalization_enabled:
    raise ValueError("This notebook expects a graph built with the frozen spatial transform.")
spatial = graph.build_config.spatial_transform_metadata
print(f"spatial transform: site={spatial.site_id}; units={spatial.coordinate_units}; "
      f"center={spatial.center}; scale={spatial.scale:.8g}; "
      f"position_weight={graph.build_config.position_weight}; "
      f"sha256={graph.build_config.spatial_transform_sha256}")
print(f"AE checkpoint SHA-256: {graph.encoder_checkpoint_sha256}")
print(f"nodes: {node_count} ({len(graph.sample_node_ids)} samples, {len(graph.ap_node_ids)} APs)")
print(f"edges: {edge_count} ({len(graph.radio_edges)} physical, {len(graph.virtual_edges)} temporary virtual)")
print(f"RSSI feature space: {graph.build_config.similarity_feature_space}")
print(
    "virtual positive rule: cosine > "
    f"{graph.build_config.virtual_edge_sigma:.3f}; "
    f"minimum shared APs={graph.build_config.minimum_shared_aps}"
)
print(
    "negative pair rule: cosine <= "
    f"{graph.build_config.negative_similarity_threshold:.3f}; "
    f"maximum per sample={graph.build_config.max_negative_pairs_per_sample}"
)
print(f"mined negative pairs: {len(graph.negative_pairs)}")
for pair in graph.negative_pairs[:5]:
    print(
        f"  {pair.left_sample_id} <-> {pair.right_sample_id}: "
        f"RSSI cosine={pair.signal_cosine_similarity:.3f}, "
        f"strength={pair.negative_strength:.3f}"
    )
print(f"feature shapes: nodes={tuple(graph_tensors.node_features.shape)}, edges={tuple(graph_tensors.edge_features.shape)}")
print(f"device: {device}")

graph: output/notebook_runs/GF/real-GF-fingerprint0-ae-v2-mse-rssi-pairs-v1-spatial-v1-softplus-cosine-t0p1-nw1-d256-z32-drop0p1/graph_state.json
dataset/batch: GF/fingerprint0
graph ID: graph:868138d5bb067bebc98a049b41a86810acc63bef0e5b6478a40035c5fef89967
normalization: paper_ae
spatial transform: site=GF; units=unknown; center=(417.8094128413554, 427.5767924937898); scale=140.68361; position_weight=1.0; sha256=f639c0b1ef66f7c12faadc837285de344cc3abda59d4b03948205d2916de1749
AE checkpoint SHA-256: a7c06f5f9af10ae09bded97ea4a9c07074498c85713d5e3b2545c38ba214c447
nodes: 1065 (392 samples, 673 APs)
edges: 21107 (20474 physical, 633 temporary virtual)
RSSI feature space: normalized_rssi_cosine_min_overlap_v1
virtual positive rule: cosine > 0.900; minimum shared APs=3
negative pair rule: cosine <= 0.500; maximum per sample=8
mined negative pairs: 1237
  scan:0001a8f3c506d5c2b93e30f338e8b6e575faa97ebc41882b9e818e6524ebf3b2 <-> scan:19d1a212d7a75d2ae44621c2afb98b9b067a04823998f14d3cf6775e64

## Model

`EdgeFeatureSAGE` implements the repository's Algorithm 1 node and edge feature updates. Each epoch processes the whole graph: node messages depend on neighboring nodes and edge vectors, so this example does not split the graph into independent shuffled rows.

In [2]:
num_layers = 2
dropout_probability = 0.1
initialization_seed = 0

model_config = GNNModelConfig(
    graph_feature_dim=graph.node_feature_dim,
    edge_feature_dim=graph.node_feature_dim,
    node_output_dim=graph.node_feature_dim,
    edge_output_dim=graph.node_feature_dim,
    num_layers=num_layers,
    dropout_probability=dropout_probability,
    initialization_seed=initialization_seed,
)
model = EdgeFeatureSAGE(model_config).to(device)
print(model)
print(f"trainable parameters: {sum(parameter.numel() for parameter in model.parameters()):,}")

EdgeFeatureSAGE(
  (W0): ParameterList(
      (0): Parameter containing: [torch.float32 of size 34x68 (cuda:0)]
      (1): Parameter containing: [torch.float32 of size 34x68 (cuda:0)]
  )
  (W1): ParameterList(
      (0): Parameter containing: [torch.float32 of size 34x68 (cuda:0)]
      (1): Parameter containing: [torch.float32 of size 34x68 (cuda:0)]
  )
  (W2): ParameterList(
      (0): Parameter containing: [torch.float32 of size 34x102 (cuda:0)]
      (1): Parameter containing: [torch.float32 of size 34x102 (cuda:0)]
  )
)
trainable parameters: 16,184


In [3]:
# --- Added: eval-mode diagnostics for sample representation collapse. ---
# Sample nodes only: AP nodes have a different role and are not included in
# the global sample-to-sample collapse statistics.
COLLAPSE_LOG_EVERY_K_EPOCHS = 1
COLLAPSE_EXACT_PAIR_LIMIT = 500
COLLAPSE_RANDOM_PAIR_COUNT = 50_000
COLLAPSE_PAIR_SEED = 0

sample_ids = tuple(graph.sample_node_ids)
node_row_by_id = {node_id: row for row, node_id in enumerate(graph_tensors.node_ids)}
sample_rows = torch.tensor(
    [node_row_by_id[node_id] for node_id in sample_ids],
    dtype=torch.long,
    device=device,
)
sample_row_by_id = {node_id: row for row, node_id in enumerate(sample_ids)}
assert len(sample_ids) == len(set(sample_ids))
assert sample_rows.shape == (len(sample_ids),)

# Keep the exact virtual-neighbor comparison separate from arbitrary sample
# pairs. A high virtual-pair cosine alone can indicate local smoothing; high
# cosine across all sample pairs is stronger evidence of global collapse.
virtual_row_pairs = set()
for edge in graph.virtual_edges:
    left_id = edge.sample_node_id_left
    right_id = edge.sample_node_id_right
    if left_id not in sample_row_by_id or right_id not in sample_row_by_id:
        raise ValueError(f"virtual edge references an unknown sample ID: {edge.edge_id}")
    left_row = sample_row_by_id[left_id]
    right_row = sample_row_by_id[right_id]
    if left_row != right_row:
        virtual_row_pairs.add(tuple(sorted((left_row, right_row))))


def representation_stats(sample_embeddings):
    """Summarize pairwise similarity and centered effective rank."""
    z = sample_embeddings.detach().to(device="cpu", dtype=torch.float32)
    if z.ndim != 2 or z.shape[0] != len(sample_ids):
        raise ValueError("collapse diagnostics need one embedding row per sample ID")
    sample_count, feature_width = z.shape
    norms = torch.linalg.vector_norm(z, dim=1)
    unit = F.normalize(z, p=2, dim=1, eps=1e-12)

    # Exact all-pair measurement for this notebook's small graph. For larger
    # graphs, use a fixed-seed pair sample to keep memory bounded.
    if sample_count <= COLLAPSE_EXACT_PAIR_LIMIT:
        pair_rows = torch.triu_indices(sample_count, sample_count, offset=1).T
        pair_mode = "all pairs"
    elif sample_count >= 2:
        generator = torch.Generator(device="cpu").manual_seed(COLLAPSE_PAIR_SEED)
        left = torch.randint(
            sample_count, (COLLAPSE_RANDOM_PAIR_COUNT,), generator=generator
        )
        right = torch.randint(
            sample_count - 1, (COLLAPSE_RANDOM_PAIR_COUNT,), generator=generator
        )
        right += right >= left
        pair_rows = torch.stack((left, right), dim=1)
        pair_mode = f"{COLLAPSE_RANDOM_PAIR_COUNT} fixed-seed sampled pairs"
    else:
        pair_rows = torch.empty((0, 2), dtype=torch.long)
        pair_mode = "no sample pairs"

    if pair_rows.numel():
        pair_cosines = (unit[pair_rows[:, 0]] * unit[pair_rows[:, 1]]).sum(dim=1)
    else:
        pair_cosines = torch.empty((0,), dtype=torch.float32)

    if pair_cosines.numel():
        cosine_mean = pair_cosines.mean().item()
        cosine_std = pair_cosines.std(unbiased=False).item()
        cosine_p90 = torch.quantile(pair_cosines, 0.90).item()
        mean_distance = (2.0 - 2.0 * pair_cosines).clamp_min(0).sqrt().mean().item()
    else:
        cosine_mean = cosine_std = cosine_p90 = mean_distance = float("nan")

    if virtual_row_pairs:
        virtual_rows = torch.tensor(sorted(virtual_row_pairs), dtype=torch.long)
        virtual_cosines = (
            unit[virtual_rows[:, 0]] * unit[virtual_rows[:, 1]]
        ).sum(dim=1)
        virtual_cosine_mean = virtual_cosines.mean().item()
    else:
        virtual_cosine_mean = float("nan")

    if pair_rows.numel():
        sampled_virtual = torch.tensor(
            [tuple(sorted((int(i), int(j)))) in virtual_row_pairs for i, j in pair_rows],
            dtype=torch.bool,
        )
        other_cosines = pair_cosines[~sampled_virtual]
        other_cosine_mean = (
            other_cosines.mean().item() if other_cosines.numel() else float("nan")
        )
    else:
        other_cosine_mean = float("nan")

    centered = z - z.mean(dim=0, keepdim=True)
    eigenvalues = torch.linalg.svdvals(centered).square()
    denominator = eigenvalues.square().sum()
    participation_ratio = (
        float((eigenvalues.sum().square() / denominator).item())
        if denominator.item() > 0.0 else 0.0
    )
    maximum_centered_rank = max(1, min(sample_count - 1, feature_width))

    return {
        "norm_mean": norms.mean().item() if norms.numel() else float("nan"),
        "zero_rows": (norms < 1e-8).float().mean().item() if norms.numel() else float("nan"),
        "pair_mode": pair_mode,
        "pair_cos_mean": cosine_mean,
        "pair_cos_std": cosine_std,
        "pair_cos_p90": cosine_p90,
        "pair_distance_mean": mean_distance,
        "virtual_pair_cos_mean": virtual_cosine_mean,
        "other_pair_cos_mean": other_cosine_mean,
        "virtual_pair_count": len(virtual_row_pairs),
        "participation_ratio": participation_ratio,
        "relative_rank": participation_ratio / maximum_centered_rank,
    }


# Interpretation: global collapse is suggested by sample-pair cosine approaching 1,
# pair distance approaching 0, and participation ratio shrinking toward 1 versus
# initialization. High virtual-pair cosine with lower other-pair cosine and a
# retained participation ratio points more toward local smoothing. No single
# cutoff is universal. The model normalizes each output row, so norm_mean should
# usually be near 1; zero_rows is the useful norm-related warning signal.
# Capture a same-graph, same-model-architecture baseline before any optimizer
# updates. Eval mode disables dropout for a fair comparison with later metrics.
model.eval()
with torch.no_grad():
    initial_sample_embeddings = model(graph_tensors).node_features[sample_rows]
initial_representation_stats = representation_stats(initial_sample_embeddings)
print("Untrained sample representation:", initial_representation_stats)
model.train()


Untrained sample representation: {'norm_mean': 1.0, 'zero_rows': 0.0, 'pair_mode': 'all pairs', 'pair_cos_mean': 0.5590570569038391, 'pair_cos_std': 0.19017784297466278, 'pair_cos_p90': 0.8146083950996399, 'pair_distance_mean': 0.9121288657188416, 'virtual_pair_cos_mean': 0.9673476219177246, 'other_pair_cos_mean': 0.5556566119194031, 'virtual_pair_count': 633, 'participation_ratio': 7.139042377471924, 'relative_rank': 0.20997183463152716}


EdgeFeatureSAGE(
  (W0): ParameterList(
      (0): Parameter containing: [torch.float32 of size 34x68 (cuda:0)]
      (1): Parameter containing: [torch.float32 of size 34x68 (cuda:0)]
  )
  (W1): ParameterList(
      (0): Parameter containing: [torch.float32 of size 34x68 (cuda:0)]
      (1): Parameter containing: [torch.float32 of size 34x68 (cuda:0)]
  )
  (W2): ParameterList(
      (0): Parameter containing: [torch.float32 of size 34x102 (cuda:0)]
      (1): Parameter containing: [torch.float32 of size 34x102 (cuda:0)]
  )
)

## Training

The graph stores one positive relation per physical radio edge and virtual RSSI-similar scan pair. It also stores bounded negative scan pairs mined where normalized RSSI cosine is at or below the configured cutoff, subject to the minimum shared-AP requirement.

For each forward pass, `equation4_loss` sums the positive edge attraction term and adds a scaled negative-pair term. The negative term compares the endpoints' learned embedding cosine against a margin, with pair strengths based on RSSI dissimilarity. The notebook calls the same function as `sage_ve.train_gnn`, so it includes the actual negative-pair loss rather than recreating only part of the objective.

The objective reads final node embeddings, so the final layer's edge-update matrix `W2` has no path to the loss; its missing gradient is expected. Earlier `W2` layers can affect later node layers.

The expanded training log reports component losses and train/eval cosine and gradient diagnostics. Rerun the training cell and final-evaluation cell to populate those fields; the saved training output may be from an earlier logging version.


In [4]:
# from pathlib import Path
# from gufu_data import load_prepared_dataset

# import numpy as np

# n = len(graph.sample_node_ids)
# X = np.asarray(graph.node_features[:n], dtype=float)

# signal = X[:, :graph.encoder_feature_dim]
# xy_input = X[:, graph.encoder_feature_dim:]

# prepared = load_prepared_dataset(
#     Path("output/notebook_runs/GF/"
#          "real-GF-fingerprint0-ae-v2-mse-rssi-pairs-v1-d256-z32-drop0p1/"
#          "prepared_bootstrap.json")
# )

# positions = {
#     p.scan_id: (p.x, p.y)
#     for p in prepared.training_reference_positions()
# }

# # Coordinates aligned to prepared.scans; NaN for scans without training labels
# coords = np.array(
#     [positions.get(scan.scan_id, (np.nan, np.nan)) for scan in prepared.scans],
#     dtype=np.float32,
# )

# print(prepared.summary())

In [5]:
# from unknot.utils.data_viz import *

# Ax = AxisGrid(1, 1, (10, 10))
# Ax[0].scatter(coords[:, 0], coords[:, 1])

In [6]:
training_config = GNNTrainingConfig(
    epochs=50,
    learning_rate=0.01,
    weight_decay=0.0,
    seed=0,
    contrastive_temperature=0.1,
    negative_weight=1.0,
)

epochs = training_config.epochs
learning_rate = training_config.learning_rate
temperature = training_config.contrastive_temperature
negative_weight = training_config.negative_weight
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=training_config.weight_decay,
)
torch.manual_seed(training_config.seed)

positive_left_rows = graph_tensors.edge_left_rows
positive_right_rows = graph_tensors.edge_right_rows
radio_edge_count = graph_tensors.radio_edge_count
virtual_edge_count = positive_left_rows.numel() - radio_edge_count
negative_left_rows = graph_tensors.negative_left_rows
negative_right_rows = graph_tensors.negative_right_rows
negative_pair_count = negative_left_rows.numel()
if positive_left_rows.numel() == 0 or negative_pair_count == 0:
    raise ValueError("Softplus cosine training requires nonempty positive and negative pairs.")


def _pair_cosines(node_embeddings, left_rows, right_rows):
    if left_rows.numel() == 0:
        return node_embeddings.new_empty((0,))
    normalized = F.normalize(node_embeddings, p=2, dim=-1)
    return (normalized[left_rows] * normalized[right_rows]).sum(dim=-1)


def _cosine_summary(name, values, *, positive=True):
    if values.numel() == 0:
        return f"{name}=none"
    low_tail = torch.quantile(values.detach(), 0.10).item()
    high_tail = torch.quantile(values.detach(), 0.90).item()
    if positive:
        threshold_fraction = (values.detach() < 0.0).float().mean().item()
        fraction_text = f"frac<0={threshold_fraction:.1%}"
    else:
        threshold_fraction = (values.detach() > 0.0).float().mean().item()
        fraction_text = f"frac>0={threshold_fraction:.1%}"
    return (
        f"{name} mean/p10/p90="
        f"{values.detach().mean().item():.3f}/"
        f"{low_tail:.3f}/{high_tail:.3f} {fraction_text}"
    )


def _gradient_l2(gradients):
    present = [gradient.detach().square().sum() for gradient in gradients if gradient is not None]
    if not present:
        return 0.0
    return torch.sqrt(torch.stack(present).sum()).item()


parameters = tuple(model.parameters())
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    output = model(graph_tensors)
    positive_loss, negative_loss = equation4_loss_components(
        output.node_features,
        graph,
        temperature=temperature,
    )
    weighted_negative_loss = negative_weight * negative_loss
    loss = positive_loss + weighted_negative_loss

    # Measure each objective component's parameter-gradient magnitude on the
    # same dropout realization before accumulating the full objective gradient.
    positive_only_gradients = torch.autograd.grad(
        positive_loss,
        parameters,
        retain_graph=True,
        allow_unused=True,
    )
    negative_only_gradients = torch.autograd.grad(
        weighted_negative_loss,
        parameters,
        retain_graph=True,
        allow_unused=True,
    )
    loss.backward()
    total_gradients = tuple(parameter.grad for parameter in parameters)

    train_positive_cosines = _pair_cosines(
        output.node_features, positive_left_rows, positive_right_rows
    )
    train_radio_cosines = train_positive_cosines[:radio_edge_count]
    train_virtual_cosines = train_positive_cosines[radio_edge_count:]
    train_negative_cosines = _pair_cosines(
        output.node_features, negative_left_rows, negative_right_rows
    )
    train_loss_value = loss.detach().item()
    train_positive_value = positive_loss.detach().item()
    train_negative_value = negative_loss.detach().item()
    train_weighted_negative_value = weighted_negative_loss.detach().item()

    gradient_by_id = {
        id(parameter): (positive_gradient, negative_gradient, total_gradient)
        for parameter, positive_gradient, negative_gradient, total_gradient in zip(
            parameters,
            positive_only_gradients,
            negative_only_gradients,
            total_gradients,
        )
    }
    layer_gradient_summary = []
    for layer in range(num_layers):
        for weight_name in ("W0", "W1", "W2"):
            parameter = getattr(model, weight_name)[layer]
            positive_gradient, negative_gradient, total_gradient = gradient_by_id[id(parameter)]
            norms = tuple(
                "none" if gradient is None
                else f"{torch.linalg.vector_norm(gradient).item():.3g}"
                for gradient in (positive_gradient, negative_gradient, total_gradient)
            )
            layer_gradient_summary.append(
                f"{weight_name}[{layer}] p/n/total={norms[0]}/{norms[1]}/{norms[2]}"
            )

    optimizer.step()

    # Report deterministic eval-mode metrics for the updated parameters as well
    # as train-mode metrics from the stochastic forward pass used for gradients.
    model.eval()
    with torch.no_grad():
        eval_output = model(graph_tensors)
        eval_positive_loss, eval_negative_loss = equation4_loss_components(
            eval_output.node_features,
            graph,
            temperature=temperature,
        )
        eval_weighted_negative_loss = negative_weight * eval_negative_loss
        eval_loss = eval_positive_loss + eval_weighted_negative_loss
        eval_positive_cosines = _pair_cosines(
            eval_output.node_features, positive_left_rows, positive_right_rows
        )
        eval_radio_cosines = eval_positive_cosines[:radio_edge_count]
        eval_virtual_cosines = eval_positive_cosines[radio_edge_count:]
        eval_negative_cosines = _pair_cosines(
            eval_output.node_features, negative_left_rows, negative_right_rows
        )
        eval_sample_embeddings = eval_output.node_features[sample_rows]
    stats = representation_stats(eval_sample_embeddings)
    baseline_rank = initial_representation_stats["participation_ratio"]
    rank_vs_initial = (
        stats["participation_ratio"] / baseline_rank
        if baseline_rank > 0.0 else float("nan")
    )

    print(
        f"epoch {epoch + 1:3d}/{epochs} | "
        f"train loss={train_loss_value:.5f} "
        f"(positive={train_positive_value:.5f}, "
        f"negative={train_negative_value:.5f}, "
        f"weighted_negative={train_weighted_negative_value:.5f})"
    )
    print(
        "  train cosine: "
        + "; ".join((
            _cosine_summary("radio+AP", train_radio_cosines),
            _cosine_summary("virtual", train_virtual_cosines),
            _cosine_summary("mined_negative", train_negative_cosines, positive=False),
        ))
    )
    print(
        f"  eval loss={eval_loss.item():.5f} "
        f"(positive={eval_positive_loss.item():.5f}, "
        f"negative={eval_negative_loss.item():.5f}, "
        f"weighted_negative={eval_weighted_negative_loss.item():.5f})"
    )
    print(
        "  eval cosine: "
        + "; ".join((
            _cosine_summary("radio+AP", eval_radio_cosines),
            _cosine_summary("virtual", eval_virtual_cosines),
            _cosine_summary("mined_negative", eval_negative_cosines, positive=False),
        ))
    )
    print(
        f"  eval samples: pair cosine mean/std/p90="
        f"{stats['pair_cos_mean']:.3f}/{stats['pair_cos_std']:.3f}/"
        f"{stats['pair_cos_p90']:.3f}; mean distance={stats['pair_distance_mean']:.3f}; "
        f"participation ratio={stats['participation_ratio']:.2f} "
        f"(vs init={rank_vs_initial:.3f}); zero rows={stats['zero_rows']:.1%}"
    )
    print(
        f"  gradient L2: positive={_gradient_l2(positive_only_gradients):.3g}, "
        f"weighted_negative={_gradient_l2(negative_only_gradients):.3g}, "
        f"total={_gradient_l2(total_gradients):.3g}; "
        + "; ".join(layer_gradient_summary)
    )
    model.train()


epoch   1/50 | train loss=5.06151 (positive=0.00762, negative=5.05389, weighted_negative=5.05389)
  train cosine: radio+AP mean/p10/p90=0.608/0.402/0.797 frac<0=0.0%; virtual mean/p10/p90=0.792/0.588/0.942 frac<0=0.0%; mined_negative mean/p10/p90=0.503/0.285/0.709 frac>0=100.0%
  eval loss=4.46268 (positive=0.00534, negative=4.45734, weighted_negative=4.45734)
  eval cosine: radio+AP mean/p10/p90=0.662/0.444/0.886 frac<0=0.0%; virtual mean/p10/p90=0.976/0.938/0.998 frac<0=0.0%; mined_negative mean/p10/p90=0.440/0.181/0.678 frac>0=99.5%
  eval samples: pair cosine mean/std/p90=0.484/0.204/0.759; mean distance=0.990; participation ratio=7.35 (vs init=1.029); zero rows=0.0%
  gradient L2: positive=0.0757, weighted_negative=9.29, total=9.22; W0[0] p/n/total=0.0259/2.95/2.93; W1[0] p/n/total=0.0443/5.35/5.32; W2[0] p/n/total=0.0159/2.53/2.52; W0[1] p/n/total=0.0285/4.06/4.03; W1[1] p/n/total=0.045/5.1/5.07; W2[1] p/n/total=none/none/none
epoch   2/50 | train loss=3.87568 (positive=0.01845, 

## Final embeddings

This final pass disables dropout and keeps the embeddings in memory, keyed by the graph's stable node IDs. No checkpoint, feature file, or log file is written.

In [10]:
model.eval()
with torch.no_grad():
    final_output = model(graph_tensors)
    final_positive_loss, final_negative_loss = equation4_loss_components(
        final_output.node_features,
        graph,
        temperature=training_config.contrastive_temperature,
    )
    final_weighted_negative_loss = (
        training_config.negative_weight * final_negative_loss
    )
    final_loss = final_positive_loss + final_weighted_negative_loss
    final_positive_cosines = _pair_cosines(
        final_output.node_features, positive_left_rows, positive_right_rows
    )
    final_radio_cosines = final_positive_cosines[:radio_edge_count]
    final_virtual_cosines = final_positive_cosines[radio_edge_count:]
    final_negative_cosines = _pair_cosines(
        final_output.node_features, negative_left_rows, negative_right_rows
    )
    final_sample_embeddings = final_output.node_features[sample_rows]

final_stats = representation_stats(final_sample_embeddings)
node_embeddings = {
    node_id: final_output.node_features[row].cpu()
    for row, node_id in enumerate(graph_tensors.node_ids)
}
assert len(node_embeddings) == node_count
assert all(vector.shape == (graph.node_feature_dim,) for vector in node_embeddings.values())
print(
    f"final eval loss={final_loss.item():.6f} "
    f"(positive={final_positive_loss.item():.6f}, "
    f"negative={final_negative_loss.item():.6f}, "
    f"weighted_negative={final_weighted_negative_loss.item():.6f})"
)
print(
    "final eval cosine: "
    + "; ".join((
        _cosine_summary("radio+AP", final_radio_cosines),
        _cosine_summary("virtual", final_virtual_cosines),
        _cosine_summary("mined_negative", final_negative_cosines, positive=False),
    ))
)
print(
    f"final sample representation: pair cosine mean/std/p90="
    f"{final_stats['pair_cos_mean']:.4f}/{final_stats['pair_cos_std']:.4f}/"
    f"{final_stats['pair_cos_p90']:.4f}; mean distance="
    f"{final_stats['pair_distance_mean']:.4f}; participation ratio="
    f"{final_stats['participation_ratio']:.2f}; zero rows={final_stats['zero_rows']:.1%}"
)
print(f"final node embedding shape=({len(node_embeddings)}, {graph.node_feature_dim})")
for node_id, vector in list(node_embeddings.items())[:3]:
    print(node_id, vector[:6].tolist())


final eval loss=0.442744 (positive=0.132054, negative=0.310690, weighted_negative=0.310690)
final eval cosine: radio+AP mean/p10/p90=0.382/0.083/0.834 frac<0=3.6%; virtual mean/p10/p90=0.962/0.891/0.996 frac<0=0.0%; mined_negative mean/p10/p90=-0.148/-0.259/-0.023 frac>0=8.4%
final sample representation: pair cosine mean/std/p90=0.0553/0.3280/0.6424; mean distance=1.3386; participation ratio=8.47; zero rows=0.0%
final node embedding shape=(1065, 34)
scan:8db92bd0ad00117d37e8391c55ba001f00b6e83cfd3d5bbe680819bee3f07bee [-0.006960997357964516, 0.43992534279823303, 0.49126607179641724, -0.04700717702507973, -0.017405249178409576, -0.05067044496536255]
scan:2b907c901662f37bfa4c0485d401c88506f96e0cccf0db202a29a94cdae98386 [0.02533837780356407, 0.5668405294418335, 0.3951236605644226, -0.04298768937587738, -0.032993245869874954, -0.05909708887338638]
scan:a380faa440dee8eae37ee948ef857aacc2bea3081e569ae1680beb969f13dabb [0.3475258946418762, 0.2749342620372772, 0.601916491985321, -0.03698380663